# Milestone 2 - NumPy Neural Network

# Milestone 2: Neural Network Foundations

## Origins and Core Idea

Biological neurons receive signals through dendrites, combine them inside the cell body, and pass a signal onward through the axon if the combined input is strong enough. Artificial neurons are a simplified mathematical version of that idea. They take input values, multiply each one by a learned weight, add a bias, and pass the result through an activation function.

The useful part is not that artificial neurons copy biology perfectly. The useful part is the idea of many simple units connected in layers. A first layer can learn simple patterns in the data, later layers can combine those patterns, and the output layer can turn the learned representation into a prediction. That layer-by-layer structure is what makes neural networks flexible.

## Components

- Neurons: small computing units that combine inputs into an output.
- Layers: groups of neurons. Input layers receive data, hidden layers transform it, and output layers make predictions.
- Weights: learned numbers that control how strongly one unit affects another.
- Biases: learned offsets that let a neuron activate even when the weighted input is not enough by itself.
- Activations: non-linear functions such as sigmoid or ReLU.

Non-linearity is essential because stacked linear layers still collapse into one linear transformation. Without a non-linear activation, the network could only draw straight-line decision boundaries. XOR is a good example: no single straight line can separate the classes, but a hidden layer with a non-linear activation can.

## Implementation

I implemented a `2 -> 4 -> 1` feedforward network. It trains on XOR using only NumPy:

- Forward pass: input data is transformed through a hidden sigmoid layer and an output sigmoid layer.
- Loss: binary cross-entropy measures the gap between predicted probability and true class.
- Backpropagation: gradients are computed from the output layer back to the hidden layer.
- Update: gradient descent changes weights and biases after each epoch.

## Reflection

Building this by hand made backpropagation feel less mysterious. The surprising part was how much depends on saving the intermediate values from the forward pass. The activation outputs are needed again during the backward pass, so the network is not just producing predictions; it is also keeping the trail needed to learn from mistakes.

## Early Hypothesis

For the capstone, I started with this hypothesis: if a dataset has repeated temporal patterns, then a neural network that receives a fixed window of past observations should learn enough seasonal structure to improve over a simple baseline. This points toward a time-series forecasting project where the input variable is recent history, the target variable is the next time step, and success is measured with forecasting error such as MAE or RMSE.


## Runnable Code:

In [9]:
"""
Milestone 2: A small neural network from scratch with NumPy.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class TrainResult:
    losses: list[float]
    predictions: np.ndarray
    accuracy: float


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Squash values into (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_backward(output: np.ndarray) -> np.ndarray:
    """Derivative of sigmoid expressed using its already-computed output."""
    return output * (1.0 - output)


def initialize_parameters(seed: int = 7) -> dict[str, np.ndarray]:
    """Create weights and biases for a 2 -> 4 -> 1 feedforward network."""
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0.0, 0.8, size=(2, 4)),
        "b1": np.zeros((1, 4)),
        "W2": rng.normal(0.0, 0.8, size=(4, 1)),
        "b2": np.zeros((1, 1)),
    }


def forward(
    x: np.ndarray, params: dict[str, np.ndarray]
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """Run one forward pass and keep intermediate values for backpropagation."""
    z1 = x @ params["W1"] + params["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    a2 = sigmoid(z2)
    return a2, {"x": x, "a1": a1, "a2": a2}


def binary_cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute average binary cross-entropy with clipping for numerical safety."""
    eps = 1e-8
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return float(-np.mean(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred)))


def backward(
    y_true: np.ndarray, cache: dict[str, np.ndarray], params: dict[str, np.ndarray]
) -> dict[str, np.ndarray]:
    """Compute gradients with the chain rule from output back to input layer."""
    m = y_true.shape[0]
    x, a1, a2 = cache["x"], cache["a1"], cache["a2"]

    dz2 = (a2 - y_true) / m
    d_w2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ params["W2"].T
    dz1 = da1 * sigmoid_backward(a1)
    d_w1 = x.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    return {"W1": d_w1, "b1": db1, "W2": d_w2, "b2": db2}


def train(epochs: int = 8000, learning_rate: float = 1.0) -> TrainResult:
    x = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
    y = np.array([[0], [1], [1], [0]], dtype=float)

    params = initialize_parameters()
    losses: list[float] = []

    for epoch in range(epochs):
        y_pred, cache = forward(x, params)
        loss = binary_cross_entropy(y, y_pred)
        grads = backward(y, cache, params)

        for key in params:
            params[key] -= learning_rate * grads[key]

        if epoch % 200 == 0 or epoch == epochs - 1:
            losses.append(loss)

    probabilities, _ = forward(x, params)
    predictions = (probabilities >= 0.5).astype(int)
    accuracy = float(np.mean(predictions == y))

    np.savez(
        ARTIFACT_DIR / "milestone2_xor_results.npz",
        probabilities=probabilities,
        predictions=predictions,
        losses=np.array(losses),
    )
    return TrainResult(losses=losses, predictions=predictions, accuracy=accuracy)



result = train()
print(f"Final sampled loss: {result.losses[-1]:.4f}")
print(f"Accuracy: {result.accuracy:.2%}")
print("Predictions for [00, 01, 10, 11]:", result.predictions.ravel().tolist())

Final sampled loss: 0.0012
Accuracy: 100.00%
Predictions for [00, 01, 10, 11]: [0, 1, 1, 0]
